### Building CNN Architecture in pytorch using the nn module

* key Steps
    * Define a model
        * Use torch.nn.Module to build CNN layers like conolutional , pooling and fully connected layers
        *  forward pass
            * Define how the input flows through the layers to produce output
        * Model Summary
            * Inspect the structure and learnable parameters
### Training and ealuating CNNs in pytorch
* Trianing 
    * Perform forward and packward passes , calculate loss and update weights using an optimizer
    * Evaluation
        * Test the model on unseen data and compute metrics like accuracy and loss
### Experimenting with CNN model design and tuning hyperparameters
* Experimentaion Areas
    * layer depth
        * Add or remove convolutional and pooling layers to observe the impact
    * filter size
    * Learning rate

In [10]:
from torchvision.datasets import CIFAR10
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [4]:
# Define Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
# Load CIFAR-10 Dataset
train_dataset = CIFAR10(root='./data',train=True, download=True, transform=transform)
test_dataset = CIFAR10(root='./data',train=False, download=True, transform=transform)
# Create DataLoaders



100%|██████████| 170M/170M [00:05<00:00, 33.2MB/s] 


In [5]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size= 64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size= 64, shuffle=False)

print("Training Data Size:", len(train_loader.dataset))
print("Testing Data Size:", len(test_loader.dataset))

Training Data Size: 50000
Testing Data Size: 10000


In [9]:
# Define a CNN model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3,6,5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,16,5)
        self.fc1 = nn.Linear(16*5*5,120)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84,10)
    
    def forward(self,x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 16*5*5)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
model = CNN()
print(model)
         

CNN(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [11]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
def train_model (model, train_loader, criterion,optimizer , epochs =10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            # zero gradient
            optimizer.zero_grad()
            # forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            # backward pass and optimize
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")
    
train_model(model, train_loader, criterion, optimizer, epochs=10)

Epoch [1/10], Loss: 1.6482
Epoch [2/10], Loss: 1.3631
Epoch [3/10], Loss: 1.2281
Epoch [4/10], Loss: 1.1373
Epoch [5/10], Loss: 1.0706
Epoch [6/10], Loss: 1.0156
Epoch [7/10], Loss: 0.9697
Epoch [8/10], Loss: 0.9240
Epoch [9/10], Loss: 0.8858
Epoch [10/10], Loss: 0.8544


In [12]:
# Evaluate on the test datasets
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs =model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")
evaluate_model(model, test_loader)


Test Accuracy: 62.32%
